<div style="text-align: center; padding: 80px 40px; border: 3px solid #1a1a2e; border-radius: 8px; background: linear-gradient(135deg, #f8f9fa 0%, #e8eef5 100%);">

<h1 style="font-size: 2.4em; color: #1a1a2e; margin-bottom: 10px; letter-spacing: 1px;">Time Series Analysis Report</h1>

<hr style="width: 40%; border: 1px solid #4a6fa5; margin: 30px auto;">

<p style="font-size: 1.2em; color: #333; margin: 8px 0;"><strong>Students:</strong> Ulysse LONG &amp; Camille PECH</p>
<p style="font-size: 1.2em; color: #333; margin: 8px 0;"><strong>Class:</strong> FGE5</p>
<p style="font-size: 1.2em; color: #333; margin: 8px 0;"><strong>School:</strong> EPF</p>

<hr style="width: 40%; border: 1px solid #4a6fa5; margin: 30px auto;">

<p style="font-size: 1.1em; color: #555; font-style: italic; margin-top: 20px;">
Google Trends time series analysis<br>
Search query « Idée de cadeau » — France (2004 – 2026)
</p>

</div>

## Table of Contents

1. [Definition of Interest (Google Trends)](#definition-of-interest-google-trends)
2. [Setting up the environment](#setting-up-the-environment)
3. [Dataset visualization](#dataset-visualization)
4. [Decomposition model — Buys-Ballot test](#decomposition-model--buys-ballot-test)

## Definition of Interest (Google Trends)

**Interest** for a search query over a given period is shown on a **relative scale from 0 to 100**, where:

- **100** corresponds to the peak interest for the time range of the chart;
- **50** means the term is half as popular as the peak;
- **0** means there is not enough data for that period.

In this report, we analyze the evolution of interest for the Google query **« Idée de cadeau »** in France, from January 2004 to October 2026.

## Setting up the environment

The cell below creates a local Python virtual environment (`.venv`), installs all required libraries, registers a Jupyter kernel for this project, and generates a `requirements.txt` file for GitHub.

**Run it once**, then select the kernel **Python (TSA-EPF)** via *Kernel ▸ Change Kernel*.  
Do **not** push the `.venv` folder to GitHub — only `requirements.txt`.

In [ ]:
# Virtual environment setup (.venv) — run once
# Do NOT push .venv to GitHub (see .gitignore)

import subprocess
import sys
from pathlib import Path

VENV_DIR = Path(".venv")
PYTHON = sys.executable
PACKAGES = [
    "pandas",
    "matplotlib",
    "numpy",
    "statsmodels",
    "scikit-learn",
    "plotly",
    "ipykernel",
]

# 1. Create the venv if it does not exist
if not VENV_DIR.exists():
    print("Creating virtual environment (.venv)...")
    subprocess.check_call([PYTHON, "-m", "venv", str(VENV_DIR)])
else:
    print(".venv already exists — reusing it.")

# Paths to the venv Python / pip (macOS / Linux / Windows)
if sys.platform == "win32":
    venv_python = VENV_DIR / "Scripts" / "python.exe"
else:
    venv_python = VENV_DIR / "bin" / "python"

# 2. Install required packages into the venv
print("Installing packages...")
subprocess.check_call(
    [str(venv_python), "-m", "pip", "install", "--upgrade", "pip"]
)
subprocess.check_call(
    [str(venv_python), "-m", "pip", "install", *PACKAGES]
)

# 3. Register a Jupyter kernel for this project
subprocess.check_call(
    [
        str(venv_python),
        "-m",
        "ipykernel",
        "install",
        "--user",
        "--name",
        "tsa-epf",
        "--display-name",
        "Python (TSA-EPF)",
    ]
)

# 4. Write a lean requirements.txt (direct deps only — not a full pip freeze)
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(PACKAGES) + "\n")

print("\n✅ Environment ready.")
print("→ Select the kernel: Kernel ▸ Change Kernel ▸ Python (TSA-EPF)")
print("→ On GitHub: push requirements.txt, ignore .venv/")

## Dataset visualization

The cell below loads the Google Trends CSV and displays an **interactive** chart of monthly interest for the query **« Idée de cadeau »** in France.

You can **zoom** (scroll or box-select), **pan**, and use the **range slider** under the chart to focus on a specific period. Double-click to reset the view.

In [3]:
import pandas as pd
import plotly.graph_objects as go

# Load the dataset
df = pd.read_csv(
    "time_series_FR_20040101-0100_20261001-1602.csv",
    parse_dates=["Time"],
)
df = df.rename(columns={"Time": "Date", "Idée de cadeau": "Interest"})
df = df.set_index("Date")

# Interactive plot (zoom, pan, period selection)
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["Interest"],
        mode="lines",
        name="Interest",
        line=dict(color="#2c5aa0", width=2),
        fill="tozeroy",
        fillcolor="rgba(44, 90, 160, 0.15)",
        hovertemplate="Date: %{x|%Y-%m}<br>Interest: %{y}<extra></extra>",
    )
)

fig.update_layout(
    title="Google Trends Interest — « Idée de cadeau » (France, 2004–2026)",
    hovermode="x unified",
    template="plotly_white",
    height=500,
    yaxis=dict(title="Interest (scale 0–100)", range=[0, 105]),
    xaxis=dict(
        title="Date",
        rangeslider=dict(visible=True),
        rangeselector=dict(
            buttons=[
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(count=5, label="5y", step="year", stepmode="backward"),
                dict(count=10, label="10y", step="year", stepmode="backward"),
                dict(step="all", label="All"),
            ]
        ),
    ),
)

fig.show()

print(f"Period: {df.index.min().strftime('%Y-%m')} → {df.index.max().strftime('%Y-%m')}")
print(f"Number of observations: {len(df)}")
print(f"Average interest: {df['Interest'].mean():.1f}")
print(f"Max interest: {df['Interest'].max()} ({df['Interest'].idxmax().strftime('%Y-%m')})")

Period: 2004-01 → 2026-09
Number of observations: 273
Average interest: 23.9
Max interest: 100 (2013-12)


## Decomposition model — Buys-Ballot test

Buys-Ballot table: each **row** = one period (year), each **column** = one season (month).

The Buys-Ballot test evaluates the linear relationship between the **periodic average** and the **periodic standard deviation**:

$$\sigma_{p_i} = \alpha + \beta\,\bar{p}_i + \varepsilon_i$$

- If $\beta$ is statistically null → **additive** model
- If $\beta$ is significantly non-null → **multiplicative** model

In [ ]:
import numpy as np
import statsmodels.api as sm

# --- Buys-Ballot table (years × months) ---
bb = df.copy()
bb["year"] = bb.index.year
bb["month"] = bb.index.month

# Keep complete periods only (12 months), as required for a full BB table
complete_years = bb.groupby("year").size()
complete_years = complete_years[complete_years == 12].index
bb = bb[bb["year"].isin(complete_years)]

table = bb.pivot_table(index="year", columns="month", values="Interest")

# Period (row) averages and standard deviations
period_avg = table.mean(axis=1)   # p̄_i
period_std = table.std(axis=1)    # σ_p_i

# Season (column) averages and standard deviations
season_avg = table.mean(axis=0)   # s̄_j
season_std = table.std(axis=0)    # σ_s_j

# Display BB table with period / seasonal averages and stds
bb_display = table.copy()
bb_display["Average (p̄_i)"] = period_avg
bb_display["StdDev (σ_p_i)"] = period_std
bb_display.loc["Average (s̄_j)"] = list(season_avg) + [np.nan, np.nan]
bb_display.loc["StdDev (σ_s_j)"] = list(season_std) + [np.nan, np.nan]

print("Buys-Ballot table")
display(bb_display.round(2))

# --- Buys-Ballot test: σ_p_i = α + β p̄_i + ε_i ---
X = sm.add_constant(period_avg)
ols = sm.OLS(period_std, X).fit()

beta = ols.params.iloc[1]
p_value = ols.pvalues.iloc[1]

print(ols.summary())
print(f"\nβ = {beta:.4f}")
print(f"p-value (β) = {p_value:.4f}")

# Decision rule from the lecture:
# - β statistically null      → additive
# - β significantly non-null → multiplicative
if p_value >= 0.05:
    decomposition_model = "additive"
    print("\n→ β is statistically null → additive model (X_t = T_t + S_t + R_t)")
else:
    decomposition_model = "multiplicative"
    print("\n→ β is significantly non-null → multiplicative model (X_t = T_t × S_t × R_t)")

print(f"\nSelected decomposition model: {decomposition_model}")